# 3. Entraînement et Évaluation des Modèles de Classification Supervisée

Dans cette section, nous appliquons plusieurs algorithmes de classification supervisée afin de prédire la recommandation d'achat des clientes (`Recommended IND`) à partir des descripteurs textuels (TF-IDF) et catégoriels prétraités :

- **Baseline** : Zero-R (`DummyClassifier`)
- **Modèles Probabilistes & Basés sur la Distance** :
  - $k$-Nearest Neighbors ($k$-NN)
  - Multinomial Naïve Bayes
- **Modèles d'Arbres & Ensemble** :
  - Decision Tree (Arbre de Décision)
  - Random Forest (Forêt Aléatoire)
- **Modèles à Marge Maximale** :
  - Support Vector Machine (Linear SVC)

Chaque modèle complexe est optimisé via **`GridSearchCV`** (Validation Croisée) et évalué à l'aide des métriques clés : **Accuracy**, **F1-Score** et **ROC-AUC**.

### 3.1. Chargement des Données et Environnement

Nous importons les matrices creuses prétraitées ($X_{train}$, $X_{test}$) ainsi que les étiquettes cibles ($y_{train}$, $y_{test}$).

In [5]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed"


X_train = sp.load_npz(DATA_PATH / "X_train_processed.npz")
X_test = sp.load_npz(DATA_PATH / "X_test_processed.npz")
y_train = pd.read_csv(DATA_PATH / "y_train.csv").values.ravel()
y_test = pd.read_csv(DATA_PATH / "y_test.csv").values.ravel()

results = []

print("✓ Setup completed and data loaded!")

✓ Setup completed and data loaded!


### 3.2. Algorithme des $k$-Plus Proches Voisins ($k$-NN)

L'algorithme k-NN est une méthode non paramétrique basée sur la mesure de distance entre les instances. 

**Hyperparamètres optimisés via `GridSearchCV` :**
* `n_neighbors` (k) : Nombre de voisins à prendre en compte ($k \in \{5, 11, 21\}$).
* `weights` : Pondération des voisins (`uniform` ou `distance`).
* `algorithm` : Recherche par force brute (`brute`), adaptée aux matrices creuses TF-IDF.

In [6]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score


PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed"

X_train = sp.load_npz(DATA_PATH / "X_train_processed.npz")
X_test = sp.load_npz(DATA_PATH / "X_test_processed.npz")
y_train = pd.read_csv(DATA_PATH / "y_train.csv").values.ravel()
y_test = pd.read_csv(DATA_PATH / "y_test.csv").values.ravel()


results = []


param_grid_knn = {
    "n_neighbors": [5, 11, 21],
    "weights": ["uniform", "distance"],
    "algorithm": ["brute"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid_knn, 
    cv=3, 
    scoring='f1', 
    n_jobs=2
)


grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

try:
    auc_knn = roc_auc_score(y_test, best_knn.predict_proba(X_test)[:, 1])
except Exception:
    auc_knn = None

results.append({
    "Model": "K-Nearest Neighbors",
    "Best Params": grid_knn.best_params_,
    "Accuracy": round(acc_knn, 4),
    "F1-Score": round(f1_knn, 4),
    "ROC-AUC": round(auc_knn, 4) if auc_knn else "N/A"
})

print("=== K-Nearest Neighbors (k-NN) ===")
print(f"Best Hyperparameters: {grid_knn.best_params_}")
print(f"Accuracy: {acc_knn:.4f} | F1-Score: {f1_knn:.4f} | ROC-AUC: {auc_knn if auc_knn else 'N/A'}\n")
print(classification_report(y_test, y_pred_knn))

=== K-Nearest Neighbors (k-NN) ===
Best Hyperparameters: {'algorithm': 'brute', 'n_neighbors': 21, 'weights': 'uniform'}
Accuracy: 0.9186 | F1-Score: 0.9519 | ROC-AUC: 0.9642336920823539

              precision    recall  f1-score   support

           0       0.87      0.64      0.74       834
           1       0.93      0.98      0.95      3859

    accuracy                           0.92      4693
   macro avg       0.90      0.81      0.84      4693
weighted avg       0.92      0.92      0.91      4693



### 3.3. Classifieur Naïve Bayes (Multinomial)

Le classifieur Naïve Bayes repose sur le théorème de Bayes avec une hypothèse forte d'indépendance conditionnelle entre les variables. Le modèle **Multinomial Naïve Bayes** est particulièrement adapté aux fréquences de mots et représentations TF-IDF.

**Traitement spécifique & Hyperparamètres :**
* Tronquage des valeurs négatives (`np.clip`) pour garantir la compatibilité avec la distribution multinomiale.
* Optimisation du paramètre de lissage de Laplace ($\alpha \in \{0.01, 0.1, 0.5, 1.0, 2.0\}$).

In [7]:
from sklearn.naive_bayes import MultinomialNB


X_tr_nb = X_train.copy()
X_tr_nb.data = np.clip(X_tr_nb.data, 0, None)

X_te_nb = X_test.copy()
X_te_nb.data = np.clip(X_te_nb.data, 0, None)


param_grid_nb = {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0]}

grid_nb = GridSearchCV(MultinomialNB(), param_grid_nb, cv=5, scoring='f1', n_jobs=-1)
grid_nb.fit(X_tr_nb, y_train)


best_nb = grid_nb.best_estimator_
y_pred_nb = best_nb.predict(X_te_nb)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, best_nb.predict_proba(X_te_nb)[:, 1])

results.append({
    "Model": "Multinomial Naive Bayes",
    "Best Params": grid_nb.best_params_,
    "Accuracy": round(acc_nb, 4),
    "F1-Score": round(f1_nb, 4),
    "ROC-AUC": round(auc_nb, 4)
})

print("=== Multinomial Naïve Bayes ===")
print(f"Best Hyperparameters: {grid_nb.best_params_}")
print(f"Accuracy: {acc_nb:.4f} | F1-Score: {f1_nb:.4f} | ROC-AUC: {auc_nb:.4f}\n")
print(classification_report(y_test, y_pred_nb))

=== Multinomial Naïve Bayes ===
Best Hyperparameters: {'alpha': 0.1}
Accuracy: 0.9052 | F1-Score: 0.9425 | ROC-AUC: 0.9531

              precision    recall  f1-score   support

           0       0.74      0.72      0.73       834
           1       0.94      0.95      0.94      3859

    accuracy                           0.91      4693
   macro avg       0.84      0.83      0.84      4693
weighted avg       0.90      0.91      0.90      4693



### 3.4. Modèle de Référence (Baseline) : Zero-R

Le modèle **Zero-R (DummyClassifier)** sert de référence minimale (*sanity check*) pour évaluer la pertinence et le gain d'apprentissage des modèles plus complexes :

- **Fonctionnement** : Il préduit de façon systématique la classe la plus fréquente (*most_frequent*) présente dans l'ensemble d'entraînement, sans prendre en compte les caractéristiques des données.
- **Utilité** : Il définit le seuil de performance de base. Tout modèle de classification digne de ce nom doit obtenir des résultats nettement supérieurs à cette baseline pour prouver son efficacité prédictive.
- **Métriques** : Évaluation via l'Accuracy et le F1-Score (la métrique ROC-AUC n'est pas applicable en l'absence de scores de probabilité déterministes).

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC

# ==========================================
# 3.4. Baseline : Zero-R (DummyClassifier)
# ==========================================
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
acc_dummy = accuracy_score(y_test, y_pred_dummy)
f1_dummy = f1_score(y_test, y_pred_dummy)

results.append({
    "Model": "Zero-R (Baseline)",
    "Best Params": {"strategy": "most_frequent"},
    "Accuracy": round(acc_dummy, 4),
    "F1-Score": round(f1_dummy, 4),
    "ROC-AUC": "N/A"
})

print("=== Zero-R (Baseline) ===")
print(f"Accuracy: {acc_dummy:.4f} | F1-Score: {f1_dummy:.4f}\n")


# ==========================================


=== Zero-R (Baseline) ===
Accuracy: 0.8223 | F1-Score: 0.9025



### 3.5. Arbre de Décision (Decision Tree)

L'**Arbre de Décision (Decision Tree)** est un modèle d'apprentissage supervisé non paramétrique qui sépare les données de manière hiérarchique en appliquant des règles de décision successives[cite: 5] :

- **Fonctionnement** : Il découpe l'espace des descripteurs en sous-ensembles homogènes en choisissant la meilleure variable de séparation à chaque nœud[cite: 5].
- **Optimisation des Hyperparamètres (`GridSearchCV`)** :
  - `criterion` : La fonction permettant de mesurer la qualité de la séparation (`gini` pour l'impureté de Gini ou `entropy` pour le gain d'information)[cite: 5].
  - `max_depth` : La profondeur maximale de l'arbre pour limiter la complexité et éviter le surapprentissage[cite: 5].
  - `min_samples_split` : Le nombre minimal d'échantillons requis dans un nœud pour effectuer une nouvelle séparation[cite: 5].
- **Évaluation** : Mesure des performances prédictives via l'Accuracy, le F1-Score et la courbe ROC-AUC[cite: 5].

In [9]:
# 3.5. Arbre de Décision (Decision Tree)
# ==========================================
param_grid_dt = {
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=3, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, best_dt.predict_proba(X_test)[:, 1])

results.append({
    "Model": "Decision Tree",
    "Best Params": grid_dt.best_params_,
    "Accuracy": round(acc_dt, 4),
    "F1-Score": round(f1_dt, 4),
    "ROC-AUC": round(auc_dt, 4)
})

print("=== Decision Tree ===")
print(f"Best Hyperparameters: {grid_dt.best_params_}")
print(f"Accuracy: {acc_dt:.4f} | F1-Score: {f1_dt:.4f} | ROC-AUC: {auc_dt:.4f}\n")
print(classification_report(y_test, y_pred_dt))


# ==========================================


=== Decision Tree ===
Best Hyperparameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 5}
Accuracy: 0.9301 | F1-Score: 0.9563 | ROC-AUC: 0.9684

              precision    recall  f1-score   support

           0       0.74      0.93      0.83       834
           1       0.98      0.93      0.96      3859

    accuracy                           0.93      4693
   macro avg       0.86      0.93      0.89      4693
weighted avg       0.94      0.93      0.93      4693



### 3.6. Forêt Aléatoire (Random Forest)

La **Forêt Aléatoire (Random Forest)** est un algorithme d'apprentissage d'ensemble basé sur l'agrégation de plusieurs arbres de décision indépendants (*Bagging*)[cite: 5] :

- **Fonctionnement** : Il construit une multitude d'arbres de décision entraînés sur des sous-ensembles aléatoires de données (*bootstrap*) et de variables (*feature subsampling*), puis combine leurs prédictions par vote de majorité pour réduire fortement la variance et prévenir le surapprentissage[cite: 5].
- **Optimisation des Hyperparamètres (`GridSearchCV`)** :
  - `n_estimators` : Le nombre d'arbres à construire dans la forêt.
  - `max_depth` : La profondeur maximale autorisée pour chaque arbre afin de contrôler la complexité du modèle.
  - `min_samples_split` : Le nombre minimal d'échantillons requis pour diviser un nœud interne.
- **Évaluation** : Mesure globale des performances à travers l'Accuracy, le F1-Score et le ROC-AUC calculé à partir des probabilités prédites (`predict_proba`).

In [ ]:

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])

results.append({
    "Model": "Random Forest",
    "Best Params": grid_rf.best_params_,
    "Accuracy": round(acc_rf, 4),
    "F1-Score": round(f1_rf, 4),
    "ROC-AUC": round(auc_rf, 4)
})

print("=== Random Forest ===")
print(f"Best Hyperparameters: {grid_rf.best_params_}")
print(f"Accuracy: {acc_rf:.4f} | F1-Score: {f1_rf:.4f} | ROC-AUC: {auc_rf:.4f}\n")
print(classification_report(y_test, y_pred_rf))


# ==========================================


=== Random Forest ===
Best Hyperparameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy: 0.9356 | F1-Score: 0.9613 | ROC-AUC: 0.9730

              precision    recall  f1-score   support

           0       0.86      0.76      0.81       834
           1       0.95      0.97      0.96      3859

    accuracy                           0.94      4693
   macro avg       0.90      0.87      0.88      4693
weighted avg       0.93      0.94      0.93      4693



### 3.7. Support Vector Machines (Linear SVC)

Les **Machines à Vecteurs de Support (SVM)** cherchent à trouver un hyperplan de séparation optimal qui maximise la marge entre les différentes classes de données. 

Dans cette section, nous utilisons **LinearSVC** pour évaluer les performances d'un classifieur SVM à noyau linéaire :
- **Fonctionnement** : Idéal pour les données textuelles à forte dimensionnalité (mots-clés / TF-IDF) grâce à sa rapidité de calcul et son efficacité linéaire.
- **Optimisation des Hyperparamètres (`GridSearchCV`)** :
  - `C` : Paramètre de régularisation contrôlant le compromis entre la largeur de la marge et la tolérance aux erreurs d'entraînement.
  - `loss` : Fonction de perte à optimiser (`hinge` pour la perte SVM standard ou `squared_hinge` pour sa version quadratique).
- **Évaluation** : Calcul de l'Accuracy, du F1-score et du ROC-AUC à l'aide de la fonction de décision (`decision_function`).

In [11]:
# 3.7. Support Vector Machine (Linear SVC & SVC)
# ==========================================
param_grid_svc = {
    "C": [0.1, 1.0, 10.0],
    "loss": ["hinge", "squared_hinge"]
}

grid_linear_svc = GridSearchCV(LinearSVC(random_state=42, max_iter=2000), param_grid_svc, cv=3, scoring='f1', n_jobs=-1)
grid_linear_svc.fit(X_train, y_train)

best_linear_svc = grid_linear_svc.best_estimator_
y_pred_linear_svc = best_linear_svc.predict(X_test)

acc_linear_svc = accuracy_score(y_test, y_pred_linear_svc)
f1_linear_svc = f1_score(y_test, y_pred_linear_svc)

try:
    auc_linear_svc = roc_auc_score(y_test, best_linear_svc.decision_function(X_test))
except Exception:
    auc_linear_svc = None

results.append({
    "Model": "Linear SVC",
    "Best Params": grid_linear_svc.best_params_,
    "Accuracy": round(acc_linear_svc, 4),
    "F1-Score": round(f1_linear_svc, 4),
    "ROC-AUC": round(auc_linear_svc, 4) if auc_linear_svc is not None else "N/A"
})

print("=== Linear SVC ===")
print(f"Best Hyperparameters: {grid_linear_svc.best_params_}")
print(f"Accuracy: {acc_linear_svc:.4f} | F1-Score: {f1_linear_svc:.4f} | ROC-AUC: {auc_linear_svc if auc_linear_svc is not None else 'N/A'}\n")
print(classification_report(y_test, y_pred_linear_svc))

=== Linear SVC ===
Best Hyperparameters: {'C': 0.1, 'loss': 'squared_hinge'}
Accuracy: 0.9378 | F1-Score: 0.9622 | ROC-AUC: 0.9774599599926175

              precision    recall  f1-score   support

           0       0.83      0.82      0.82       834
           1       0.96      0.96      0.96      3859

    accuracy                           0.94      4693
   macro avg       0.90      0.89      0.89      4693
weighted avg       0.94      0.94      0.94      4693



### 3.8. Synthèse et Comparaison des Modèles

Le tableau ci-dessous récapitule les meilleures configurations d'hyperparamètres ainsi que les performances obtenues sur l'ensemble de test en termes d'**Accuracy**, **F1-Score** et **ROC-AUC**.

In [12]:

comparison_df = pd.DataFrame(results)
display(comparison_df.sort_values(by=['F1-Score', 'ROC-AUC'], ascending=False).reset_index(drop=True))

,Model,Best Params,Accuracy,F1-Score,ROC-AUC
0,Linear SVC,"{'C': 0.1, 'loss': 'squared_hinge'}",0.9378,0.9622,0.9775
1,Random Forest,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.9356,0.9613,0.973
2,Decision Tree,"{'criterion': 'gini', 'max_depth': 5, 'min_sam...",0.9301,0.9563,0.9684
3,K-Nearest Neighbors,"{'algorithm': 'brute', 'n_neighbors': 21, 'wei...",0.9186,0.9519,0.9642
4,Multinomial Naive Bayes,{'alpha': 0.1},0.9052,0.9425,0.9531
5,Zero-R (Baseline),{'strategy': 'most_frequent'},0.8223,0.9025,N/A


### 3.9. Sauvegarde et Sérialisation du Meilleur Modèle

Une fois la phase d'évaluation et de comparaison des performances terminée, le modèle ayant obtenu les meilleurs résultats est prêt pour le déploiement :

- **Fonctionnement** : Utilisation de la bibliothèque `joblib` pour sérialiser l'objet Python correspondant au meilleur modèle (`best_estimator_`) et enregistrer ses paramètres appris sous forme d'un fichier binaire (`.pkl`).
- **Gestion des Répertoires** : Création automatique d'un dossier dédié `/models` à la racine du projet à l'aide de `pathlib.Path`.
- **Réutilisation / Déploiement** : Ce fichier sérialisé `best_model.pkl` pourra ensuite être directement chargé dans d'autres scripts ou applications (par exemple un dashboard interactif **Streamlit** ou une API **FastAPI**) pour effectuer des prédictions en temps réel sur de nouvelles données.

In [13]:
import joblib
from pathlib import Path

ROOT_DIR = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Sélection et sauvegarde du meilleur modèle entraîné (ex: grid_rf.best_estimator_)
best_model = grid_rf.best_estimator_ 

joblib.dump(best_model, MODELS_DIR / "best_model.pkl")
print("✅ best_model.pkl créé avec succès dans models/")

✅ best_model.pkl créé avec succès dans models/
